# FULL PIPELINE — Job Market Analyzer for Career Readiness in South Africa

## This notebook combines all notebooks 00-13 into one sequential pipeline.

## How to use:
1. Run all cells from top to bottom (Runtime → Run all)
2. Each section corresponds to a notebook in the pipeline
3. Outputs are saved for each stage

## Pipeline:
00 Project Setup → 01 Data Understanding → 02 Data Collection → 03 Data Cleaning → 04 EDA → 05 NLP → 06 Features → 07 Resume → 08 Similarity → 09 Matching → 10 ML → 11 Readiness → 12 Market → 13 Dashboard

---

# NOTEBOOK 00 — PROJECT SETUP

In [ ]:
# STEP 1 — Define Project Configuration
PROJECT_NAME = "Job Market Analyzer for Career Readiness in South Africa"
PROJECT_VERSION = "1.0.0"
AUTHOR = "Nbody-01"
DESCRIPTION = "A Data Science system to analyze South African job market data and provide career readiness insights for graduates."

print(f"Project: {PROJECT_NAME}")
print(f"Version: {PROJECT_VERSION}")
print(f"Author: {AUTHOR}")
print(f"Description: {DESCRIPTION}")

In [ ]:
# STEP 2 — Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import joblib

print("All libraries imported successfully.")

In [ ]:
# STEP 3 — Configure Paths
DATA_RAW = "data/raw/"
DATA_INTERIM = "data/interim/"
DATA_PROCESSED = "data/processed/"
OUTPUTS = "outputs/"
MODELS = "models/"

for path in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, OUTPUTS, MODELS]:
    os.makedirs(path, exist_ok=True)

print("Paths configured.")

In [ ]:
# STEP 4 — Set Random Seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Random seed set to: {RANDOM_SEED}")

In [ ]:
# STEP 5 — Configure Reproducibility
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
print("Reproducibility settings configured.")

In [ ]:
# STEP 6 — Load Utility Functions
def save_dataframe(df, filepath, index=False):
    df.to_csv(filepath, index=index)
    print(f"Saved: {filepath}")

def load_dataframe(filepath):
    df = pd.read_csv(filepath)
    print(f"Loaded: {filepath} | Shape: {df.shape}")
    return df

def calculate_cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

print("Utility functions loaded.")

In [ ]:
# STEP 7 — Verify Environment
print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print("Environment ready.")

---

# NOTEBOOK 01 — DATA UNDERSTANDING

In [ ]:
# STEP 2 — Load the Dataset
df = pd.read_csv('data/raw/sa_it_jobs.csv')
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
# STEP 3-4 — Inspect Shape and Data Types
print(f"\nData Types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# STEP 5 — Analyze Missing Values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Missing Values Report:")
print(missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False))

In [ ]:
# STEP 6 — Analyze Duplicates
full_duplicates = df.duplicated().sum()
id_duplicates = df.duplicated(subset=['Job_ID']).sum()
desc_duplicates = df.duplicated(subset=['Job_Description']).sum()
print(f"Full row duplicates: {full_duplicates}")
print(f"Job_ID duplicates: {id_duplicates}")
print(f"Job_Description duplicates: {desc_duplicates}")

In [ ]:
# STEP 7-9 — Inspect Unique Values, Text Fields, Job Titles and Locations
print(f"\nUnique Job Titles: {df['Job_Title'].nunique()}")
print(f"Unique Companies: {df['Company'].nunique()}")
print(f"Unique Provinces: {df['Province'].nunique()}")
print(f"\nTop 10 Job Titles:")
print(df['Job_Title'].value_counts().head(10))
print(f"\nProvince Distribution:")
print(df['Province'].value_counts())

---

# NOTEBOOK 02 — DATA COLLECTION

In [ ]:
# STEP 2-3 — Data Sources and Field Completeness
print(f"Total records: {len(df)}")
print(f"\nField Completeness:")
for col in df.columns:
    fill_rate = (1 - df[col].isnull().sum() / len(df)) * 100
    print(f"  {col:20s}: {fill_rate:.1f}% filled")

In [ ]:
# STEP 5 — Source Coverage
print("\nProvince Coverage:")
for prov, count in df['Province'].value_counts().items():
    pct = count / len(df) * 100
    print(f"  {prov:<20}: {count:>4} jobs ({pct:>5.1f}%)")
print(f"\nUnique companies: {df['Company'].nunique()}")

In [ ]:
# STEP 8 — Save Raw Dataset with Metadata
df.to_csv('data/raw/sa_it_jobs_raw.csv', index=False)
metadata = {
    'collection_date': '2024',
    'source': 'South African online job boards and recruitment sites',
    'total_records': len(df),
    'fields_collected': list(df.columns),
    'provinces_covered': df['Province'].nunique(),
    'companies_covered': df['Company'].nunique(),
    'industries_covered': df['Industry'].nunique(),
    'known_issues': {
        'full_duplicates': int(df.duplicated().sum()),
        'missing_values': int(df.isnull().sum().sum()),
        'description_duplicates': int(df.duplicated(subset=['Job_Description']).sum())
    }
}
with open('data/raw/collection_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("\nRaw dataset and metadata saved.")

---

# NOTEBOOK 03 — DATA CLEANING

In [ ]:
# STEP 2 — Load Raw Data
df_raw = pd.read_csv('data/raw/sa_it_jobs.csv')
df = df_raw.copy()
print(f"Raw dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# STEP 3 — Remove Duplicates
print("Duplicate Removal:")
initial_count = len(df)
id_dups = df.duplicated(subset=['Job_ID']).sum()
df = df.drop_duplicates(subset=['Job_ID'], keep='first')
print(f"  Removed {id_dups} Job_ID duplicates")
desc_dups = df.duplicated(subset=['Job_Description']).sum()
df = df.drop_duplicates(subset=['Job_Description'], keep='first')
print(f"  Removed {desc_dups} Job_Description duplicates")
total_removed = initial_count - len(df)
print(f"  Total rows removed: {total_removed}")
print(f"  Rows remaining: {len(df)}")

In [ ]:
# STEP 4 — Handle Missing Values
print("\nMissing Value Treatment:")
critical_fields = ['Job_Title', 'Job_Description', 'Skills']
for field in critical_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        df = df.dropna(subset=[field])
        print(f"  Removed {missing} rows with missing {field}")
important_fields = ['Company', 'Province', 'City', 'Industry']
for field in important_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        df[field] = df[field].fillna('Unknown')
        print(f"  Filled {missing} missing {field} with 'Unknown'")
optional_fields = ['Degree_Field', 'Remote', 'Contract_Type', 'Employment_Type']
for field in optional_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        df[field] = df[field].fillna('Not specified')
        print(f"  Filled {missing} missing {field} with 'Not specified'")
exp_missing = df['Experience_Years'].isnull().sum()
if exp_missing > 0:
    median_exp = df['Experience_Years'].median()
    df['Experience_Years'] = df['Experience_Years'].fillna(median_exp)
    print(f"  Filled {exp_missing} missing Experience_Years with median ({median_exp})")
print(f"\nRows remaining: {len(df)}")

In [ ]:
# STEP 5 — Clean HTML and Unwanted Text
def clean_text(text):
    if pd.isna(text):
        return text
    text = re.sub(r'<[^>]+>', '', str(text))
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s.,;:!?()\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Job_Description'] = df['Job_Description'].apply(clean_text)
df['Job_Title'] = df['Job_Title'].apply(clean_text)
df['Company'] = df['Company'].apply(clean_text)
df['Skills'] = df['Skills'].apply(clean_text)
print("Text cleaning completed.")

In [ ]:
# STEP 6 — Normalize Job Titles
def normalize_job_title(title):
    if pd.isna(title):
        return title
    title = str(title).strip().lower()
    if 'data scientist' in title or 'data science' in title:
        return 'Data Scientist'
    elif 'data analyst' in title or 'data analytics' in title:
        return 'Data Analyst'
    elif 'data engineer' in title:
        return 'Data Engineer'
    elif 'machine learning' in title or 'ml engineer' in title:
        return 'Machine Learning Engineer'
    elif 'software developer' in title or 'software engineer' in title:
        return 'Software Developer'
    elif 'business analyst' in title or 'bi analyst' in title:
        return 'Business Analyst'
    elif 'devops' in title:
        return 'DevOps Engineer'
    elif 'cloud' in title:
        return 'Cloud Engineer'
    elif 'junior' in title or 'entry' in title or 'graduate' in title:
        return 'Junior/Entry Level'
    elif 'senior' in title or 'lead' in title or 'principal' in title:
        return 'Senior/Lead'
    else:
        return str(title).strip().title()

df['Job_Title_Normalized'] = df['Job_Title'].apply(normalize_job_title)
print("Job title normalization completed.")
print(df['Job_Title_Normalized'].value_counts().head(10))

In [ ]:
# STEP 7 — Normalize Locations
province_mapping = {
    'gauteng': 'Gauteng', 'western cape': 'Western Cape',
    'kwazulu-natal': 'KwaZulu-Natal', 'kzn': 'KwaZulu-Natal',
    'eastern cape': 'Eastern Cape', 'free state': 'Free State',
    'northern cape': 'Northern Cape', 'limpopo': 'Limpopo',
    'mpumalanga': 'Mpumalanga', 'north west': 'North West', 'northwest': 'North West'
}
def normalize_province(province):
    if pd.isna(province):
        return 'Unknown'
    province = str(province).strip().lower()
    return province_mapping.get(province, str(province).strip().title())

df['Province'] = df['Province'].apply(normalize_province)
print("Province normalization completed.")

In [ ]:
# STEP 8 — Normalize Skills
def normalize_skills(skills):
    if pd.isna(skills):
        return skills
    skill_list = [s.strip().lower() for s in str(skills).split(',')]
    skill_list = [s for s in skill_list if s]
    return ', '.join(skill_list)

df['Skills'] = df['Skills'].apply(normalize_skills)
print("Skills normalization completed.")

In [ ]:
# STEP 9 — Standardize Categories
employment_mapping = {'full-time': 'Full-Time', 'full time': 'Full-Time', 'part-time': 'Part-Time', 'part time': 'Part-Time', 'contract': 'Contract', 'permanent': 'Permanent', 'temporary': 'Temporary'}
df['Employment_Type'] = df['Employment_Type'].apply(lambda x: employment_mapping.get(str(x).strip().lower(), str(x).strip().title()) if pd.notna(x) else 'Not specified')

contract_mapping = {'permanent': 'Permanent', 'contract': 'Contract', 'temporary': 'Temporary', 'freelance': 'Freelance'}
df['Contract_Type'] = df['Contract_Type'].apply(lambda x: contract_mapping.get(str(x).strip().lower(), str(x).strip().title()) if pd.notna(x) else 'Not specified')

remote_mapping = {'yes': 'Yes', 'no': 'No', 'hybrid': 'Hybrid', 'remote': 'Remote', 'on-site': 'On-site', 'onsite': 'On-site'}
df['Remote'] = df['Remote'].apply(lambda x: remote_mapping.get(str(x).strip().lower(), str(x).strip().title()) if pd.notna(x) else 'Not specified')
print("Category standardization completed.")

In [ ]:
# STEP 10-12 — Validate and Save Clean Dataset
print(f"\nCleaned dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate Job_IDs: {df.duplicated(subset=['Job_ID']).sum()}")
df.to_csv('data/interim/sa_it_jobs_clean.csv', index=False)
print("Clean dataset saved to: data/interim/sa_it_jobs_clean.csv")

---

# NOTEBOOK 04 — EXPLORATORY DATA ANALYSIS

In [ ]:
# STEP 2 — Load Clean Dataset
df = pd.read_csv('data/interim/sa_it_jobs_clean.csv')
print(f"Clean dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# STEP 3 — Job Demand by Industry
plt.figure(figsize=(12, 6))
industry_counts = df['Industry'].value_counts().head(10)
sns.barplot(x=industry_counts.values, y=industry_counts.index, palette='viridis')
plt.title('Top 10 Industries by Job Demand', fontsize=14, fontweight='bold')
plt.xlabel('Number of Job Postings')
plt.tight_layout()
plt.savefig('outputs/eda_job_demand_by_industry.png', dpi=300, bbox_inches='tight')
plt.show()
print("Top 10 Industries:")
print(industry_counts)

In [ ]:
# STEP 4 — Top Job Titles
plt.figure(figsize=(12, 6))
title_counts = df['Job_Title_Normalized'].value_counts().head(15)
sns.barplot(x=title_counts.values, y=title_counts.index, palette='magma')
plt.title('Top 15 Job Titles in South African IT Market', fontsize=14, fontweight='bold')
plt.xlabel('Number of Job Postings')
plt.tight_layout()
plt.savefig('outputs/eda_job_titles.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 5 — Top Skills
all_skills = []
for skills in df['Skills'].dropna():
    all_skills.extend([s.strip() for s in str(skills).split(',')])
skill_counts = pd.Series(all_skills).value_counts().head(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=skill_counts.values, y=skill_counts.index, palette='coolwarm')
plt.title('Top 20 Most Demanded Skills in South African IT Market', fontsize=14, fontweight='bold')
plt.xlabel('Number of Job Postings')
plt.tight_layout()
plt.savefig('outputs/eda_top_skills.png', dpi=300, bbox_inches='tight')
plt.show()
print("Top 20 Skills:")
print(skill_counts)

In [ ]:
# STEP 6 — Province Distribution
plt.figure(figsize=(12, 6))
province_counts = df['Province'].value_counts()
sns.barplot(x=province_counts.index, y=province_counts.values, palette='Set2')
plt.title('Job Distribution by South African Province', fontsize=14, fontweight='bold')
plt.xlabel('Province')
plt.ylabel('Number of Job Postings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/eda_province_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("Province Distribution:")
print(province_counts)

In [ ]:
# STEP 7 — Work Type Distribution
plt.figure(figsize=(10, 6))
work_type_counts = df['Remote'].value_counts()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
plt.pie(work_type_counts.values, labels=work_type_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('Work Type Distribution (Remote/Hybrid/On-site)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_work_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 8 — Contract Type Distribution
plt.figure(figsize=(10, 6))
contract_counts = df['Contract_Type'].value_counts()
sns.barplot(x=contract_counts.index, y=contract_counts.values, palette='pastel')
plt.title('Contract Type Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Contract Type')
plt.ylabel('Number of Job Postings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/eda_contract_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 9 — Education Distribution
plt.figure(figsize=(10, 6))
edu_counts = df['Education'].value_counts()
sns.barplot(x=edu_counts.index, y=edu_counts.values, palette='muted')
plt.title('Education Requirements in South African IT Jobs', fontsize=14, fontweight='bold')
plt.xlabel('Education Level')
plt.ylabel('Number of Job Postings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('outputs/eda_education_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 10 — Experience Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['Experience_Years'].dropna(), bins=20, kde=True, color='skyblue')
plt.title('Experience Requirements Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Years of Experience')
plt.ylabel('Number of Job Postings')
plt.tight_layout()
plt.savefig('outputs/eda_experience_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("Experience Statistics:")
print(df['Experience_Years'].describe())

In [ ]:
# STEP 13 — Save EDA Outputs
eda_summary = {
    'total_jobs': len(df),
    'top_skills': skill_counts.head(20).to_dict(),
    'province_distribution': province_counts.to_dict(),
    'work_type_distribution': work_type_counts.to_dict(),
    'contract_type_distribution': contract_counts.to_dict(),
    'education_distribution': edu_counts.to_dict(),
    'experience_stats': df['Experience_Years'].describe().to_dict()
}
with open('outputs/eda_summary.json', 'w') as f:
    json.dump(eda_summary, f, indent=2)
print("EDA summary saved to: outputs/eda_summary.json")

---

# NOTEBOOK 05 — NLP PREPROCESSING

In [ ]:
# STEP 2 — Load Clean Dataset
df = pd.read_csv('data/interim/sa_it_jobs_clean.csv')
print(f"Dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# STEP 4 — Clean Text
def clean_text_nlp(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Job_Description_Clean'] = df['Job_Description'].apply(clean_text_nlp)
df['Skills_Clean'] = df['Skills'].apply(clean_text_nlp)
print("Text cleaning completed.")

In [ ]:
# STEP 5 — Tokenize
df['desc_tokens'] = df['Job_Description_Clean'].apply(lambda x: x.split())
df['skills_tokens'] = df['Skills_Clean'].apply(lambda x: x.split())
print("Tokenization completed.")

In [ ]:
# STEP 6 — Handle Stopwords
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
custom_stopwords = {'job', 'role', 'position', 'candidate', 'ideal', 'working', 'work',
                    'ability', 'including', 'include', 'within', 'across', 'related',
                    'etc', 'per', 'based', 'using', 'use', 'used', 'well', 'also'}
stop_words.update(custom_stopwords)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in stop_words and len(token) > 2]

df['desc_tokens_clean'] = df['desc_tokens'].apply(remove_stopwords)
df['skills_tokens_clean'] = df['skills_tokens'].apply(remove_stopwords)
print("Stopword removal completed.")

In [ ]:
# STEP 7 — Normalize Text (Lemmatization)
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]

df['desc_tokens_lemmatized'] = df['desc_tokens_clean'].apply(lemmatize_tokens)
df['skills_tokens_lemmatized'] = df['skills_tokens_clean'].apply(lemmatize_tokens)

df['Job_Description_Processed'] = df['desc_tokens_lemmatized'].apply(lambda x: ' '.join(x))
df['Skills_Processed'] = df['skills_tokens_lemmatized'].apply(lambda x: ' '.join(x))
print("Lemmatization completed.")

In [ ]:
# STEP 10 — Save Processed Text + TF-IDF Top Terms
df.to_csv('data/interim/sa_it_jobs_processed.csv', index=False)
print(f"Processed dataset saved: {df.shape[0]} rows x {df.shape[1]} columns")

tfidf = TfidfVectorizer(max_features=100, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Job_Description_Processed'])
feature_names = tfidf.get_feature_names_out()
mean_tfidf = tfidf_matrix.mean(axis=0).A1
tfidf_scores = pd.Series(mean_tfidf, index=feature_names).sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=tfidf_scores.values, y=tfidf_scores.index, palette='viridis')
plt.title('Top 20 Terms by TF-IDF Score', fontsize=14, fontweight='bold')
plt.xlabel('Mean TF-IDF Score')
plt.tight_layout()
plt.savefig('outputs/nlp_tfidf_top_terms.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nTop 20 TF-IDF Terms:")
print(tfidf_scores)

---

# NOTEBOOK 06 — FEATURE ENGINEERING

In [ ]:
# STEP 2 — Load Processed Data
df = pd.read_csv('data/interim/sa_it_jobs_processed.csv')
print(f"Processed dataset loaded: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# STEP 4 — Create Text Features
df['desc_length'] = df['Job_Description'].str.len()
df['desc_word_count'] = df['Job_Description_Processed'].apply(lambda x: len(str(x).split()))
df['has_python'] = df['Skills'].str.contains('python', case=False, na=False).astype(int)
df['has_sql'] = df['Skills'].str.contains('sql', case=False, na=False).astype(int)
df['has_cloud'] = df['Skills'].str.contains('cloud|aws|azure|gcp', case=False, na=False).astype(int)
df['has_ml'] = df['Skills'].str.contains('machine learning|ml|ai', case=False, na=False).astype(int)
print("Text features created.")

In [ ]:
# STEP 5 — Create Skill Features
df['skill_count'] = df['Skills'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0)
df['has_programming'] = df['Skills'].str.contains('python|java|javascript|c\+\+|c#|r|scala|go', case=False, na=False).astype(int)
df['has_database'] = df['Skills'].str.contains('sql|mysql|postgresql|mongodb|oracle|database', case=False, na=False).astype(int)
df['has_viz'] = df['Skills'].str.contains('tableau|power bi|powerbi|matplotlib|seaborn|plotly|dashboard', case=False, na=False).astype(int)
df['has_cloud_tech'] = df['Skills'].str.contains('aws|azure|gcp|cloud|docker|kubernetes', case=False, na=False).astype(int)
print("Skill features created.")

In [ ]:
# STEP 6 — Create Education Features
education_mapping = {
    'Not specified': 0, 'National Senior Certificate': 1, 'Higher Certificate': 2,
    'Diploma': 3, 'Advanced Diploma': 4, "Bachelor's": 5, 'Honours': 6,
    "Master's": 7, 'Doctoral': 8
}
df['education_level'] = df['Education'].map(education_mapping).fillna(0)
print("Education features created.")

In [ ]:
# STEP 7 — Create Experience Features
df['is_entry_level'] = (df['Experience_Years'] <= 1).astype(int)
df['is_mid_level'] = ((df['Experience_Years'] > 1) & (df['Experience_Years'] <= 5)).astype(int)
df['is_senior'] = (df['Experience_Years'] > 5).astype(int)
print("Experience features created.")

In [ ]:
# STEP 8 — Create Preference Features
df['is_remote'] = df['Remote'].isin(['Yes', 'Hybrid', 'Remote']).astype(int)
df['is_fulltime'] = (df['Employment_Type'] == 'Full-Time').astype(int)
df['is_permanent'] = (df['Contract_Type'] == 'Permanent').astype(int)
df['is_sme'] = (df['Company_Size'] == 'SME').astype(int)
df['is_large'] = (df['Company_Size'] == 'Large').astype(int)
print("Preference features created.")

In [ ]:
# STEP 12 — Save Feature Dataset
df.to_csv('data/processed/sa_it_jobs_features.csv', index=False)
print(f"Feature dataset saved: {df.shape[0]} rows x {df.shape[1]} columns")

---

# STATISTICAL ANALYSIS (from Notebook 06)

In [ ]:
# STEP 13 — Chi-Square + Cramér's V Analysis
categorical_pairs = [
    ('Province', 'Remote'), ('Province', 'Employment_Type'),
    ('Education', 'Experience_Years'), ('Company_Size', 'Remote'),
    ('Industry', 'Employment_Type'), ('Contract_Type', 'Remote')
]

print("Chi-Square + Cramér's V Analysis:")
print("="*60)
results = []
for var1, var2 in categorical_pairs:
    contingency_table = pd.crosstab(df[var1], df[var2])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    n = contingency_table.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * min(contingency_table.shape[0]-1, contingency_table.shape[1]-1)))
    results.append({'Variable 1': var1, 'Variable 2': var2, 'Chi2': round(chi2, 2), 'p-value': round(p_value, 4), 'Cramers V': round(cramers_v, 3), 'Significant': 'Yes' if p_value < 0.05 else 'No'})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

plt.figure(figsize=(10, 8))
pivot = results_df.pivot(index='Variable 1', columns='Variable 2', values='Cramers V')
sns.heatmap(pivot, annot=True, cmap='YlOrRd', fmt='.3f', cbar_kws={'label': "Cramér's V"})
plt.title("Cramér's V Heatmap: Feature Associations", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/stats_cramers_v_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 14 — Conditional Probability Analysis
print("Conditional Probability Analysis:")
print("="*60)
remote_by_province = pd.crosstab(df['Province'], df['Remote'], normalize='index')
print("\nP(Remote | Province):")
print(remote_by_province.round(3))

plt.figure(figsize=(12, 6))
sns.heatmap(remote_by_province, annot=True, cmap='Blues', fmt='.2f')
plt.title('P(Remote | Province)', fontsize=14, fontweight='bold')
plt.xlabel('Remote Status')
plt.ylabel('Province')
plt.tight_layout()
plt.savefig('outputs/stats_conditional_probability.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 15 — Apriori Association Rule Mining
transactions = df['Skills'].apply(lambda x: [s.strip() for s in str(x).split(',') if s.strip()])
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_apriori = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_apriori, min_support=0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("\nApriori Analysis - Frequent Itemsets:")
print("="*60)
print(f"Found {len(frequent_itemsets)} frequent itemsets")
print(f"\nTop 10:")
print(frequent_itemsets.sort_values('support', ascending=False).head(10))

if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)
    print(f"\nAssociation Rules (confidence >= 0.5): {len(rules)} rules")
    if len(rules) > 0:
        print(rules.sort_values('confidence', ascending=False).head(10))

In [ ]:
# STEP 16 — Mutual Information Analysis
mi_features = ['skill_count', 'desc_length', 'desc_word_count', 'education_level',
                'Experience_Years', 'is_remote', 'is_fulltime', 'is_permanent',
                'is_sme', 'is_large', 'is_entry_level', 'is_mid_level', 'is_senior',
                'has_python', 'has_sql', 'has_cloud', 'has_ml', 'has_programming',
                'has_database', 'has_viz', 'has_cloud_tech']

X = df[mi_features].fillna(0)
y_remote = df['is_remote']
mi_remote = mutual_info_classif(X, y_remote, random_state=42)
mi_remote_series = pd.Series(mi_remote, index=mi_features).sort_values(ascending=False)

print("\nMutual Information - Predicting Remote Status:")
print("="*60)
print(mi_remote_series)

plt.figure(figsize=(12, 8))
mi_remote_series.plot(kind='barh', color='steelblue')
plt.title('Mutual Information: Features vs Remote Status', fontsize=14, fontweight='bold')
plt.xlabel('Mutual Information Score')
plt.tight_layout()
plt.savefig('outputs/stats_mutual_information.png', dpi=300, bbox_inches='tight')
plt.show()

---

# NOTEBOOK 07 — RESUME PARSER

In [ ]:
# STEP 2 — Sample CV
sample_cv = """
JOHN DOE
Johannesburg, South Africa
john.doe@email.com | 082 123 4567

EDUCATION
BSc Data Science
University of Cape Town, 2020-2023

TECHNICAL SKILLS
Programming: Python, SQL, R
Tools: Power BI, Tableau, Excel
Machine Learning: scikit-learn, TensorFlow
Cloud: AWS, Azure

EXPERIENCE
Data Analyst Intern | ABC Company | Jan 2023 - Present
- Developed dashboards using Power BI
- Analyzed customer data using Python and SQL
- Created reports for management

CERTIFICATIONS
- AWS Certified Cloud Practitioner
- Microsoft Power BI Data Analyst

PROJECTS
Job Market Analyzer
- Built a job recommendation system using Python and NLP
- Technologies: Python, scikit-learn, Streamlit
"""
print("Sample CV loaded.")

In [ ]:
# STEP 5-10 — Extract Information
def extract_skills(cv_text):
    skill_dict = {
        'programming': ['python', 'java', 'javascript', 'c++', 'c#', 'r', 'scala', 'go', 'ruby', 'php', 'swift', 'kotlin'],
        'databases': ['sql', 'mysql', 'postgresql', 'mongodb', 'oracle', 'sqlite', 'redis', 'elasticsearch'],
        'ml_ai': ['machine learning', 'deep learning', 'tensorflow', 'pytorch', 'scikit-learn', 'keras', 'nlp', 'computer vision'],
        'cloud': ['aws', 'azure', 'gcp', 'google cloud', 'docker', 'kubernetes', 'terraform'],
        'viz': ['tableau', 'power bi', 'powerbi', 'matplotlib', 'seaborn', 'plotly', 'dashboards'],
        'tools': ['excel', 'git', 'jira', 'confluence', 'slack', 'trello'],
        'stats': ['statistics', 'regression', 'hypothesis testing', 'a/b testing', 'probability']
    }
    found_skills = set()
    cv_lower = cv_text.lower()
    for category, skills in skill_dict.items():
        for skill in skills:
            if skill in cv_lower:
                found_skills.add(skill)
    return list(found_skills)

def extract_education(cv_text):
    education = []
    qualification_patterns = [
        r'(?:BSc|Bachelor(?:\'s)?|BTech|BA)\s+(?:of\s+)?(?:Science\s+)?(?:in\s+)?[A-Za-z\s]+',
        r'(?:MSc|Master(?:\'s)?|MTech|MA)\s+(?:of\s+)?(?:Science\s+)?(?:in\s+)?[A-Za-z\s]+',
        r'(?:PhD|Doctor(?:\'s)?|Doctoral)\s+(?:of\s+)?(?:Philosophy\s+)?(?:in\s+)?[A-Za-z\s]+',
        r'(?:Diploma|Higher Certificate|Advanced Diploma)\s+(?:in\s+)?[A-Za-z\s]+',
        r'National Senior Certificate'
    ]
    for pattern in qualification_patterns:
        matches = re.findall(pattern, cv_text, re.IGNORECASE)
        education.extend(matches)
    return education

def extract_certifications(cv_text):
    certifications = []
    cert_keywords = ['aws certified', 'azure certified', 'google cloud certified', 'microsoft certified', 'cisco certified', 'comptia', 'power bi', 'tableau certified', 'sas certified', 'pmp', 'scrum master', 'itil']
    cv_lower = cv_text.lower()
    for cert in cert_keywords:
        if cert in cv_lower:
            certifications.append(cert)
    return certifications

extracted_skills = extract_skills(sample_cv)
extracted_education = extract_education(sample_cv)
extracted_certs = extract_certifications(sample_cv)

print(f"Skills: {extracted_skills}")
print(f"Education: {extracted_education}")
print(f"Certifications: {extracted_certs}")

In [ ]:
# STEP 11-14 — Build and Save Candidate Profile
candidate_profile = {
    'skills': extracted_skills,
    'education': extracted_education,
    'experience': ['Data Analyst Intern | ABC Company | Jan 2023 - Present'],
    'certifications': extracted_certs,
    'projects': ['Job Market Analyzer - Built a job recommendation system using Python and NLP']
}

with open('data/processed/candidate_profile.json', 'w') as f:
    json.dump(candidate_profile, f, indent=2)
print("Candidate profile saved to: data/processed/candidate_profile.json")

---

# NOTEBOOK 08 — TF-IDF & SIMILARITY

In [ ]:
# STEP 2-3 — Load Data
with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)
df = pd.read_csv('data/processed/sa_it_jobs_features.csv')
candidate_text = ' '.join(candidate['skills']) + ' ' + ' '.join(candidate['education']) + ' ' + ' '.join(candidate['projects'])
print(f"Candidate text: {len(candidate_text)} characters")
print(f"Jobs: {len(df)}")

In [ ]:
# STEP 4-7 — TF-IDF and Cosine Similarity
df['job_text'] = df['Job_Description_Processed'].fillna('') + ' ' + df['Skills_Processed'].fillna('')

tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.95)
tfidf_vectorizer.fit(df['job_text'])

job_tfidf = tfidf_vectorizer.transform(df['job_text'])
candidate_tfidf = tfidf_vectorizer.transform([candidate_text])

similarity_scores = cosine_similarity(candidate_tfidf, job_tfidf).flatten()
df['tfidf_similarity'] = similarity_scores
df['tfidf_similarity_score'] = (df['tfidf_similarity'] * 100).round(2)

print("Similarity Score Statistics:")
print(df['tfidf_similarity_score'].describe())
print(f"\nTop 5 Most Similar Jobs:")
print(df.nlargest(5, 'tfidf_similarity_score')[['Job_Title', 'Company', 'Province', 'tfidf_similarity_score']].to_string(index=False))

In [ ]:
# STEP 8 — Visualize Similarity Distribution
plt.figure(figsize=(12, 6))
sns.histplot(df['tfidf_similarity_score'], bins=30, kde=True, color='skyblue')
plt.title('Distribution of TF-IDF Similarity Scores', fontsize=14, fontweight='bold')
plt.xlabel('Similarity Score (0-100)')
plt.ylabel('Number of Jobs')
plt.axvline(df['tfidf_similarity_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["tfidf_similarity_score"].mean():.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/similarity_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 10 — Save Similarity Features
df.to_csv('data/processed/sa_it_jobs_with_similarity.csv', index=False)
print(f"Dataset with similarity saved: {df.shape[0]} rows x {df.shape[1]} columns")

---

# NOTEBOOK 09 — JOB MATCHING

In [ ]:
# STEP 2-3 — Load Data
with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)
df = pd.read_csv('data/processed/sa_it_jobs_with_similarity.csv')
candidate_skills = set(s.lower() for s in candidate['skills'])
print(f"Candidate skills: {candidate_skills}")
print(f"Jobs: {len(df)}")

In [ ]:
# STEP 4-5 — Compare Skills and Calculate Skill Match
def extract_job_skills(skills_str):
    if pd.isna(skills_str):
        return set()
    return set(s.strip().lower() for s in str(skills_str).split(','))

df['job_skills'] = df['Skills'].apply(extract_job_skills)

def compare_skills(job_skills, candidate_skills):
    matched = job_skills & candidate_skills
    missing = job_skills - candidate_skills
    return matched, missing

df['matched_skills'] = df['job_skills'].apply(lambda x: compare_skills(x, candidate_skills)[0])
df['missing_skills'] = df['job_skills'].apply(lambda x: compare_skills(x, candidate_skills)[1])

def calculate_skill_match(row):
    total_skills = len(row['job_skills'])
    if total_skills == 0:
        return 0
    matched = len(row['matched_skills'])
    return (matched / total_skills) * 100

df['skill_match'] = df.apply(calculate_skill_match, axis=1).round(2)
print("Skill match calculated.")

In [ ]:
# STEP 6-10 — Calculate Other Components
df['text_similarity'] = df['tfidf_similarity_score']

candidate_experience = 0
def calculate_experience_match(required_years):
    if pd.isna(required_years) or required_years == 0:
        return 100
    return min(candidate_experience / required_years, 1) * 100
df['experience_match'] = df['Experience_Years'].apply(calculate_experience_match).round(2)

candidate_education_level = 5
def calculate_education_match(required_level):
    if pd.isna(required_level):
        return 100
    if candidate_education_level >= required_level:
        return 100
    return (candidate_education_level / required_level) * 100
df['education_match'] = df['education_level'].apply(calculate_education_match).round(2)

target_role = "data analyst"
def calculate_role_relevance(job_title):
    if pd.isna(job_title):
        return 0
    job_title_lower = str(job_title).lower()
    if target_role in job_title_lower:
        return 100
    related_terms = ['data', 'analyst', 'analytics', 'bi', 'business intelligence']
    matches = sum(1 for term in related_terms if term in job_title_lower)
    return min(matches * 20, 80)
df['role_relevance'] = df['Job_Title_Normalized'].apply(calculate_role_relevance).round(2)

def calculate_preference_alignment(row):
    score = 0
    checks = 0
    checks += 1
    if 'gauteng' in str(row['Province']).lower():
        score += 1
    checks += 1
    if 'hybrid' in str(row['Remote']).lower():
        score += 1
    checks += 1
    if 'full-time' in str(row['Employment_Type']).lower():
        score += 1
    return (score / checks * 100) if checks > 0 else 50
df['preference_alignment'] = df.apply(calculate_preference_alignment, axis=1).round(2)

print("All components calculated.")

In [ ]:
# STEP 11-12 — Calculate Final Match Score
weights = {'skill_match': 0.40, 'education_match': 0.15, 'experience_match': 0.15, 'role_relevance': 0.10, 'text_similarity': 0.20}

df['match_score'] = (
    df['skill_match'] * weights['skill_match'] +
    df['education_match'] * weights['education_match'] +
    df['experience_match'] * weights['experience_match'] +
    df['role_relevance'] * weights['role_relevance'] +
    df['text_similarity'] * weights['text_similarity']
).round(2)

print("Match Score Statistics:")
print(df['match_score'].describe())
print(f"\nTop 10 Matching Jobs:")
print(df.nlargest(10, 'match_score')[['Job_Title', 'Company', 'Province', 'match_score', 'skill_match', 'text_similarity']].to_string(index=False))

In [ ]:
# STEP 13-15 — Identify Skills and Explain Match
def explain_match(row):
    explanations = []
    if row['skill_match'] >= 80:
        explanations.append("Strong skill match")
    elif row['skill_match'] >= 50:
        explanations.append("Moderate skill match")
    else:
        explanations.append("Skill gaps identified")
    if row['education_match'] == 100:
        explanations.append("Education requirement met")
    if row['experience_match'] == 100:
        explanations.append("Experience requirement met")
    if row['text_similarity'] >= 50:
        explanations.append("Good text similarity")
    return '; '.join(explanations)

df['match_explanation'] = df.apply(explain_match, axis=1)

# Visualize match score distribution
plt.figure(figsize=(12, 6))
sns.histplot(df['match_score'], bins=30, kde=True, color='green')
plt.title('Distribution of Job Match Scores', fontsize=14, fontweight='bold')
plt.xlabel('Match Score (0-100)')
plt.ylabel('Number of Jobs')
plt.axvline(df['match_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["match_score"].mean():.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/match_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 18 — Save Matching Results
df.to_csv('data/processed/sa_it_jobs_matched.csv', index=False)
print(f"Matching results saved: {df.shape[0]} rows x {df.shape[1]} columns")

---

# NOTEBOOK 10 — MACHINE LEARNING

In [ ]:
# STEP 2-3 — Inspect Labels and Create Synthetic Target
df = pd.read_csv('data/processed/sa_it_jobs_matched.csv')
match_threshold = 50
df['is_good_match'] = (df['match_score'] >= match_threshold).astype(int)
print(f"Synthetic target created (FOR DEMONSTRATION ONLY):")
print(f"Good matches: {df['is_good_match'].sum()} ({df['is_good_match'].mean()*100:.1f}%)")
print(f"Poor matches: {(df['is_good_match']==0).sum()} ({(1-df['is_good_match'].mean())*100:.1f}%)")

In [ ]:
# STEP 4-6 — Prepare Features and Split Data
feature_cols = ['skill_count', 'desc_length', 'desc_word_count', 'education_level',
                'Experience_Years', 'is_remote', 'is_fulltime', 'is_permanent',
                'is_sme', 'is_large', 'is_entry_level', 'is_mid_level', 'is_senior',
                'has_python', 'has_sql', 'has_cloud', 'has_ml', 'has_programming',
                'has_database', 'has_viz', 'has_cloud_tech']

X = df[feature_cols].fillna(0)
y = df['is_good_match']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")

In [ ]:
# STEP 7-11 — Train and Evaluate Models
# Baseline
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)

# Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)

# Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_precision = precision_score(y_test, nb_pred)
nb_recall = recall_score(y_test, nb_pred)
nb_f1 = f1_score(y_test, nb_pred)

print("All models trained and evaluated.")

In [ ]:
# STEP 13-15 — Compare Models and Select Best
model_comparison = pd.DataFrame({
    'Model': ['Baseline', 'Random Forest', 'Logistic Regression', 'Naive Bayes'],
    'Accuracy': [baseline_accuracy, rf_accuracy, lr_accuracy, nb_accuracy],
    'Precision': [0, rf_precision, lr_precision, nb_precision],
    'Recall': [0, rf_recall, lr_recall, nb_recall],
    'F1-Score': [0, rf_f1, lr_f1, nb_f1]
})

print("Model Comparison:")
print(model_comparison.to_string(index=False))

best_model_idx = model_comparison['F1-Score'].idxmax()
best_model_name = model_comparison.loc[best_model_idx, 'Model']
best_f1 = model_comparison.loc[best_model_idx, 'F1-Score']
print(f"\nBest Model: {best_model_name} (F1: {best_f1:.4f})")
print("\nWARNING: This is for demonstration only. The target is not a legitimate label.")

In [ ]:
# STEP 17 — Save Model
joblib.dump(rf_model, 'models/random_forest_model.pkl')
model_comparison.to_csv('outputs/model_comparison.csv', index=False)
print("Model and comparison saved.")

---

# NOTEBOOK 11 — CAREER READINESS

In [ ]:
# STEP 2-8 — Calculate All Components
with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)
df = pd.read_csv('data/processed/sa_it_jobs_matched.csv')

# Skill Readiness
all_skills = []
for skills in df['Skills'].dropna():
    all_skills.extend([s.strip().lower() for s in str(skills).split(',')])
skill_demand = pd.Series(all_skills).value_counts().head(20)
demanded_skills = set(skill_demand.index)
candidate_skills = set(s.lower() for s in candidate['skills'])
matched_demanded = demanded_skills & candidate_skills
skill_readiness = (len(matched_demanded) / len(demanded_skills)) * 100 if demanded_skills else 0

# Experience Readiness
candidate_experience = 0
avg_required_experience = df['Experience_Years'].mean()
experience_readiness = min(candidate_experience / avg_required_experience, 1) * 100 if avg_required_experience > 0 else 100

# Education Readiness
candidate_education_level = 5
avg_required_education = df['education_level'].mean()
education_readiness = min(candidate_education_level / avg_required_education, 1) * 100 if avg_required_education > 0 else 100

# Certification Readiness
candidate_certs = candidate.get('certifications', [])
certification_readiness = min(len(candidate_certs) / 3, 1) * 100

# Project Readiness
candidate_projects = candidate.get('projects', [])
project_readiness = min(len(candidate_projects) / 2, 1) * 100

# Market Alignment
market_alignment = df['match_score'].mean()

print(f"Skill Readiness: {skill_readiness:.1f}%")
print(f"Experience Readiness: {experience_readiness:.1f}%")
print(f"Education Readiness: {education_readiness:.1f}%")
print(f"Certification Readiness: {certification_readiness:.1f}%")
print(f"Project Readiness: {project_readiness:.1f}%")
print(f"Market Alignment: {market_alignment:.1f}%")

In [ ]:
# STEP 9-10 — Combine Components
weights = {'skill_readiness': 0.40, 'experience_readiness': 0.20, 'education_readiness': 0.15, 'certification_readiness': 0.10, 'project_readiness': 0.10, 'market_alignment': 0.05}
components = {'skill_readiness': skill_readiness, 'experience_readiness': experience_readiness, 'education_readiness': education_readiness, 'certification_readiness': certification_readiness, 'project_readiness': project_readiness, 'market_alignment': market_alignment}

career_readiness = sum(components[k] * weights[k] for k in weights)

print(f"\nCareer Readiness Score: {career_readiness:.1f}%")

# Visualize components
plt.figure(figsize=(10, 6))
components_df = pd.DataFrame({'Component': list(components.keys()), 'Score': list(components.values())})
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
plt.barh(components_df['Component'], components_df['Score'], color=colors)
plt.title('Career Readiness Components', fontsize=14, fontweight='bold')
plt.xlabel('Score (0-100)')
plt.xlim(0, 100)
plt.tight_layout()
plt.savefig('outputs/career_readiness_components.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 12-15 — Identify Strengths/Gaps and Save
strengths = [name for name, score in components.items() if score >= 70]
gaps = [name for name, score in components.items() if score < 50]

readiness_results = {
    'career_readiness_score': round(career_readiness, 2),
    'components': {k: round(v, 2) for k, v in components.items()},
    'weights': weights,
    'strengths': strengths,
    'gaps': gaps
}

with open('outputs/career_readiness.json', 'w') as f:
    json.dump(readiness_results, f, indent=2)
print(f"\nStrengths: {strengths}")
print(f"Gaps: {gaps}")
print("Career readiness results saved.")

---

# NOTEBOOK 12 — JOB MARKET ALIGNMENT

In [ ]:
# STEP 2-5 — Identify Relevant Jobs and Extract Market Skills
with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)
df = pd.read_csv('data/processed/sa_it_jobs_matched.csv')

target_role = "data analyst"
market_segment = df[df['Job_Title_Normalized'].str.contains(target_role, case=False, na=False)]
print(f"Relevant jobs for '{target_role}': {len(market_segment)}")

all_market_skills = []
for skills in market_segment['Skills'].dropna():
    all_market_skills.extend([s.strip().lower() for s in str(skills).split(',')])
market_skill_counts = pd.Series(all_market_skills).value_counts()
total_jobs = len(market_segment)

skill_demand = {}
for skill in market_skill_counts.index:
    count = market_segment['Skills'].str.contains(skill, case=False, na=False).sum()
    skill_demand[skill] = (count / total_jobs) * 100

print(f"\nTop 10 market skills:")
for skill, demand in sorted(skill_demand.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {skill}: {demand:.1f}%")

In [ ]:
# STEP 6-10 — Compare Candidate Skills and Calculate Alignment
candidate_skills = set(s.lower() for s in candidate['skills'])
demanded_skills = set(skill_demand.keys())
matched_skills = demanded_skills & candidate_skills
missing_skills = demanded_skills - candidate_skills

total_demand_weight = sum(skill_demand.values())
matched_demand_weight = sum(skill_demand.get(skill, 0) for skill in matched_skills)
market_alignment = (matched_demand_weight / total_demand_weight * 100) if total_demand_weight > 0 else 0

print(f"\nMarket Alignment: {market_alignment:.1f}%")
print(f"Matched skills: {len(matched_skills)}")
print(f"Missing skills: {len(missing_skills)}")

# Identify skill gaps
skill_gaps = []
for skill in missing_skills:
    demand = skill_demand.get(skill, 0)
    skill_gaps.append({'Skill': skill, 'Demand %': demand})
skill_gaps_df = pd.DataFrame(skill_gaps).sort_values('Demand %', ascending=False)

print(f"\nTop 5 Skill Gaps:")
print(skill_gaps_df.head(5).to_string(index=False))

# Visualize skill gaps
plt.figure(figsize=(12, 6))
top_gaps = skill_gaps_df.head(10)
sns.barplot(x='Demand %', y='Skill', data=top_gaps, palette='Reds')
plt.title('Top 10 Market Skill Gaps', fontsize=14, fontweight='bold')
plt.xlabel('Market Demand %')
plt.tight_layout()
plt.savefig('outputs/market_skill_gaps.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 13 — Save Market Analysis
priorities = skill_gaps_df.head(5)['Skill'].tolist()

market_analysis = {
    'market_alignment_score': round(market_alignment, 2),
    'target_role': target_role,
    'relevant_jobs': len(market_segment),
    'total_market': len(df),
    'matched_skills': list(matched_skills),
    'missing_skills': list(missing_skills),
    'skill_gaps': skill_gaps_df.head(10).to_dict('records'),
    'priorities': priorities,
    'province_distribution': market_segment['Province'].value_counts().to_dict(),
    'work_type_distribution': market_segment['Remote'].value_counts().to_dict(),
    'contract_type_distribution': market_segment['Contract_Type'].value_counts().to_dict()
}

with open('outputs/market_alignment.json', 'w') as f:
    json.dump(market_analysis, f, indent=2)
print("Market analysis saved.")

---

# NOTEBOOK 13 — DASHBOARD & FINAL INTEGRATION

## NOTE: This section contains the Streamlit app code.
## To run the dashboard, save this as app.py and run: streamlit run app.py
## The code below demonstrates the dashboard structure.

In [ ]:
# STEP 1 — Load All Components
import json

df = pd.read_csv('data/processed/sa_it_jobs_matched.csv')

with open('data/processed/candidate_profile.json', 'r') as f:
    candidate = json.load(f)

with open('outputs/career_readiness.json', 'r') as f:
    readiness = json.load(f)

with open('outputs/market_alignment.json', 'r') as f:
    market = json.load(f)

print("All components loaded successfully.")
print(f"Jobs: {len(df)}")
print(f"Candidate skills: {len(candidate['skills'])}")
print(f"Career readiness: {readiness['career_readiness_score']}%")
print(f"Market alignment: {market['market_alignment_score']}%")

In [ ]:
# STEP 2-12 — Dashboard Structure (Streamlit code)
# This code would be saved as app.py and run with: streamlit run app.py

dashboard_code = '''
import streamlit as st
import pandas as pd
import json

st.set_page_config(page_title="Job Market Analyzer SA", page_icon="📊", layout="wide")
st.title("Job Market Analyzer for Career Readiness in South Africa")

# Load data
df = pd.read_csv("data/processed/sa_it_jobs_matched.csv")

# Sidebar - Profile
st.sidebar.header("Your Profile")
name = st.sidebar.text_input("Name")
location = st.sidebar.selectbox("Location", ["Gauteng", "Western Cape", "KwaZulu-Natal", "Eastern Cape", "Free State", "Northern Cape", "Limpopo", "Mpumalanga", "North West"])
education = st.sidebar.selectbox("Highest Qualification", ["National Senior Certificate", "Higher Certificate", "Diploma", "Advanced Diploma", "Bachelor's", "Honours", "Master's", "Doctoral"])
experience = st.sidebar.number_input("Years of Experience", min_value=0, max_value=50, value=0)
skills_input = st.sidebar.text_area("Enter your skills (comma-separated)")
target_role = st.sidebar.text_input("Target Role (e.g., Data Analyst)")

# Main content
st.header("Dashboard")

# Career Readiness
st.subheader("Career Readiness")
st.progress(readiness["career_readiness_score"] / 100)
st.write(f"Career Readiness Score: {readiness["career_readiness_score"]:.1f}%")

# Market Alignment
st.subheader("Job Market Alignment")
st.progress(market["market_alignment_score"] / 100)
st.write(f"Market Alignment Score: {market["market_alignment_score"]:.1f}%")

# Top Job Recommendations
st.subheader("Top Job Recommendations")
top_jobs = df.nlargest(5, "match_score")
for i, job in top_jobs.iterrows():
    st.write(f"{i+1}. {job["Job_Title"]} at {job["Company"]} - Match: {job["match_score"]:.1f}%")

# Skill Gaps
st.subheader("Skill Gaps")
for gap in market["skill_gaps"][:5]:
    st.write(f"- {gap["Skill"]}: {gap["Demand %"]:.1f}% demand")
'''

print("Dashboard code prepared.")
print("To run: save as app.py and execute 'streamlit run app.py'")

In [ ]:
# STEP 13-15 — Validate and Summarize
print("="*60)
print("  FULL PIPELINE COMPLETE")
print("="*60)
print(f"\nTotal jobs processed: {len(df)}")
print(f"Career Readiness: {readiness['career_readiness_score']:.1f}%")
print(f"Market Alignment: {market['market_alignment_score']:.1f}%")
print(f"\nTop 5 Recommended Jobs:")
for i, job in df.nlargest(5, 'match_score').iterrows():
    print(f"  {i+1}. {job['Job_Title']} at {job['Company']} - Match: {job['match_score']:.1f}%")
print(f"\nTop 5 Skill Gaps:")
for gap in market['skill_gaps'][:5]:
    print(f"  - {gap['Skill']}: {gap['Demand %']:.1f}% demand")
print(f"\nAll outputs saved to: outputs/")
print("All models saved to: models/")
print("All data saved to: data/processed/")